# Irrigated vs rainfed cropland — Sentinel-1 / Sentinel-2 monthly features

**Goal.** The conventional baseline: classify parcels from a year of monthly Sentinel-2 reflectance, vegetation indices and Sentinel-1 backscatter, under the same spatial protocol as the foundation-model notebooks.

**Data.** Open STAC catalogue of [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/), read with `pystac-client` + `odc-stac` into `xarray` / `dask` (no Earth Engine):

- `sentinel-2-l2a`: B02–B12 surface reflectance, SCL cloud mask → monthly median, plus NDVI, NDWI and EVI
- `sentinel-1-rtc`: terrain-corrected γ⁰ VV and VH → monthly median in dB, plus the VH/VV ratio

12 months × 16 variables = 192 features per parcel, for the 2024 calendar year (same year as the AlphaEarth run).

**Workflow**
1. Load the labelled parcels and spatial folds
2. Build monthly composites block by block and average them per parcel (resumable cache)
3. Check cloud gaps and the seasonal profiles of both classes
4. Tune, test and cross-validate three classifiers (shared protocol in `irrmap/evaluation.py`)
5. Which variables and months matter
6. Save results to `outputs/models/s1s2_monthly_2024/`

In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import numpy as np
import odc.geo.xr  # noqa: F401  (registers the .odc accessor)
import odc.stac
import pandas as pd
import planetary_computer
import pystac_client
import xarray as xr
from odc.geo.geobox import GeoBox
from odc.geo.geom import box
from odc.geo.xr import xr_zeros
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from irrmap.evaluation import (HEADLINE, block_bootstrap, best_params, cross_validate, plot_confusion,
                               plot_cv_comparison, predict, tune_on_validation)
from irrmap.features import block_bbox, extract_by_block, parcel_zonal_mean

pd.set_option("display.width", 200)

In [ ]:
ROOT = Path.cwd()
SPLIT_FILE = ROOT / "outputs" / "split" / "parcels_split.geojson"

YEAR = 2024
RUN_NAME = f"s1s2_monthly_{YEAR}"
FEATURE_FILE = ROOT / "outputs" / "features" / f"{RUN_NAME}.parquet"
CACHE_DIR = ROOT / "outputs" / "features" / "_cache" / RUN_NAME          # one parquet per 20 km block
MODEL_DIR = ROOT / "outputs" / "models" / RUN_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
S2_COLLECTION, S1_COLLECTION = "sentinel-2-l2a", "sentinel-1-rtc"
S2_BANDS = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
S2_INDICES = ["NDVI", "NDWI", "EVI"]
S1_VARS = ["VV", "VH", "VH_VV"]
VARIABLES = S2_BANDS + S2_INDICES + S1_VARS
SCL_CLEAR = [4, 5, 6, 7]          # vegetation, bare soil, water, unclassified; clouds, shadows, snow are dropped
S2_OFFSET = 1000                  # BOA_ADD_OFFSET for processing baseline >= 04.00 (all 2024 scenes)
MAX_SCENE_CLOUD = 80              # skip almost fully cloudy scenes before loading
WORK_CRS, RESOLUTION = "EPSG:32629", 10
MONTHS = pd.date_range(f"{YEAR}-01-01", periods=12, freq="MS")

TEST_FOLD, VAL_FOLD = 0, 1
SEED = 42
N_BOOT = 1000

## 1. Reference parcels and spatial folds

Parcels and folds come from `2_split_train_test.ipynb`: 20 km blocks in 5 folds, fold 0 = test, fold 1 = validation, folds 2–4 = train.

In [ ]:
parcels = gpd.read_file(SPLIT_FILE)
parcels = parcels[["parcel_id", "label", "class_name", "block_id", "fold", "random_fold", "split", "geometry"]]
pd.crosstab(parcels["split"], parcels["class_name"], margins=True).reindex(["train", "val", "test", "All"])

## 2. Monthly composites per parcel

For each 20 km block:

1. one STAC search per collection for the whole year (Planetary Computer signs the asset URLs);
2. for each month, load only that month's scenes onto a fixed 10 m UTM grid around the block's parcels, mask clouds with SCL, and take the per-pixel median;
3. average every variable over the pixels inside each parcel.

Working one month at a time keeps memory to a few hundred MB per block. Months without a clear observation stay NaN and are imputed inside the model pipelines (fitted on training data only).

In [ ]:
def search(collection: str, bbox, **kwargs) -> list:
    return list(catalog.search(collections=[collection], bbox=bbox,
                               datetime=f"{YEAR}-01-01/{YEAR}-12-31", **kwargs).items())


def in_month(items: list, month: pd.Timestamp) -> list:
    end = month + pd.offsets.MonthBegin(1)
    return [it for it in items if month <= pd.Timestamp(it.datetime).tz_convert(None) < end]


def empty(gbox: GeoBox, names: list[str]) -> xr.Dataset:
    nan = xr_zeros(gbox, dtype="float32") * np.nan
    return xr.Dataset({n: nan for n in names})


def s2_composite(items: list, gbox: GeoBox) -> xr.Dataset:
    """Cloud-masked monthly median reflectance and indices."""
    if not items:
        return empty(gbox, S2_BANDS + S2_INDICES)
    baselines = {float(it.properties.get("s2:processing_baseline", "99")) for it in items}
    assert min(baselines) >= 4.0, f"pre-04.00 baseline found {baselines}: the offset would be wrong"
    ds = odc.stac.load(items, bands=S2_BANDS + ["SCL"], geobox=gbox, groupby="solar_day",
                       chunks={"x": 1024, "y": 1024}, resampling={"SCL": "nearest", "*": "bilinear"})
    valid = ds["SCL"].isin(SCL_CLEAR) & (ds["B04"] > 0)
    refl = ((ds[S2_BANDS].astype("float32") - S2_OFFSET) / 10_000).where(valid)
    med = refl.median("time")
    b2, b4, b8, b11 = med["B02"], med["B04"], med["B08"], med["B11"]
    med["NDVI"] = (b8 - b4) / (b8 + b4)
    med["NDWI"] = (b8 - b11) / (b8 + b11)
    med["EVI"] = 2.5 * (b8 - b4) / (b8 + 6 * b4 - 7.5 * b2 + 1)
    return med


def s1_composite(items: list, gbox: GeoBox) -> xr.Dataset:
    """Monthly median γ⁰ backscatter in dB (median taken in linear power)."""
    if not items:
        return empty(gbox, S1_VARS)
    ds = odc.stac.load(items, bands=["vv", "vh"], geobox=gbox, groupby="solar_day",
                       chunks={"x": 1024, "y": 1024}, resampling="bilinear")
    med = ds.where(ds > 0).median("time")
    out = xr.Dataset({"VV": 10 * np.log10(med["vv"]), "VH": 10 * np.log10(med["vh"])})
    out["VH_VV"] = out["VH"] - out["VV"]
    return out


def s1s2_block_features(group: gpd.GeoDataFrame) -> pd.DataFrame:
    bbox = block_bbox(group)
    gbox = GeoBox.from_geopolygon(box(*bbox, crs="EPSG:4326").to_crs(WORK_CRS), resolution=RESOLUTION)
    s2_items = search(S2_COLLECTION, bbox, query={"eo:cloud_cover": {"lt": MAX_SCENE_CLOUD}})
    s1_items = search(S1_COLLECTION, bbox)
    monthly = []
    for month in MONTHS:
        stack = xr.merge([s2_composite(in_month(s2_items, month), gbox),
                          s1_composite(in_month(s1_items, month), gbox)])[VARIABLES]
        stack = stack.to_array("feature").compute()
        stack = stack.assign_coords(feature=[f"{v}_m{month.month:02d}" for v in VARIABLES])
        table = parcel_zonal_mean(stack, group)
        monthly.append(table if not monthly else table.drop(columns="n_px"))
    return pd.concat(monthly, axis=1)

In [ ]:
if FEATURE_FILE.exists():
    features = pd.read_parquet(FEATURE_FILE)
    print(f"Loaded cached features: {features.shape}")
else:
    # Planetary Computer signs asset URLs at search time, so each block gets fresh tokens.
    catalog = pystac_client.Client.open(STAC_URL, modifier=planetary_computer.sign_inplace)
    features, failed = extract_by_block(parcels, s1s2_block_features, CACHE_DIR)
    if failed:
        print(f"{len(failed)} blocks failed — re-run this cell to retry them: {failed}")
    else:
        FEATURE_FILE.parent.mkdir(parents=True, exist_ok=True)
        features.to_parquet(FEATURE_FILE)
        print(f"Saved {FEATURE_FILE.relative_to(ROOT)}: {features.shape}")

## 3. Feature checks

In [ ]:
FEATURES = [f"{v}_m{m:02d}" for m in range(1, 13) for v in VARIABLES]
data = (parcels.drop(columns="geometry").set_index("parcel_id")
               .join(features, how="inner"))
no_signal = data[FEATURES].isna().all(axis=1)
print(f"Parcels with features: {len(data):,} of {len(parcels):,}; without any valid pixel: {no_signal.sum()}")
data = data[~no_signal]
print("Pixels per parcel:", data["n_px"].describe(percentiles=[0.05, 0.5, 0.95]).round(0).to_dict())

In [ ]:
# Cloud gaps: share of parcels without a clear Sentinel-2 observation in each month.
gaps = pd.Series({m: data[f"NDVI_m{m:02d}"].isna().mean() for m in range(1, 13)})
fig, ax = plt.subplots(figsize=(8, 3), dpi=120)
ax.bar(MONTHS.strftime("%b"), 100 * gaps, color="#9e9e9e")
ax.set_ylabel("Parcels without clear S2 (%)")
ax.set_title("Monthly Sentinel-2 gaps", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
# Seasonal profiles: median and inter-quartile range per class. Irrigated fields should stay green
# into the dry season (May–September) when rainfed cereals have been harvested.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=120)
for ax, var, unit in zip(axes, ["NDVI", "NDWI", "VH"], ["", "", " (dB)"]):
    cols = [f"{var}_m{m:02d}" for m in range(1, 13)]
    for lab, name, color in [(1, "Irrigated", "#1f77b4"), (2, "Rainfed", "#d62728")]:
        q = data.loc[data["label"] == lab, cols].quantile([0.25, 0.5, 0.75])
        ax.plot(MONTHS.strftime("%b"), q.loc[0.5], color=color, label=name, linewidth=2)
        ax.fill_between(MONTHS.strftime("%b"), q.loc[0.25], q.loc[0.75], color=color, alpha=0.15)
    ax.set_title(f"{var}{unit}", fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False)
fig.suptitle(f"Monthly profiles {YEAR}: median and IQR per class", fontweight="bold")
fig.tight_layout()
fig.savefig(MODEL_DIR / "seasonal_profiles.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Modelling protocol

| Step | Train on | Evaluate on | Purpose |
|---|---|---|---|
| Tuning | folds 2–4 | fold 1 (validation) | choose hyper-parameters by macro-F1 |
| Final test | folds 1–4 | fold 0 (test) | headline scores, block-bootstrap 95% CI |
| Spatial 5-fold CV | 4 folds | the 5th | stability across areas (mean ± std) |
| Random 5-fold CV | 4 folds | the 5th | same protocol on a random split, to show how much it overstates accuracy |

Three classifiers span the usual range:

- **Logistic regression**: a linear reference; it shows how much of the separation is already linear in the monthly features.
- **Random forest**: the conventional baseline of the proposal.
- **Histogram gradient boosting**: a strong tabular learner that handles missing values natively.

Macro-F1 is the selection metric: with 14% rainfed parcels, overall accuracy rewards ignoring the minority class.

In [ ]:
def build_logreg(C=1.0):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C, class_weight="balanced", max_iter=5000))

def build_rf(max_features="sqrt", min_samples_leaf=1, class_weight=None):
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestClassifier(n_estimators=500, max_features=max_features,
                                                min_samples_leaf=min_samples_leaf, class_weight=class_weight,
                                                n_jobs=-1, random_state=SEED))

def build_hgb(learning_rate=0.1, max_leaf_nodes=31, class_weight=None):
    # Early stopping would carve a random (non-spatial) validation split, so it is switched off.
    return HistGradientBoostingClassifier(learning_rate=learning_rate, max_leaf_nodes=max_leaf_nodes,
                                          max_iter=300, early_stopping=False, class_weight=class_weight,
                                          random_state=SEED)

MODELS = {
    "logreg": (build_logreg, {"C": [0.01, 0.1, 1.0, 10.0]}),
    "rf": (build_rf, {"max_features": ["sqrt", 0.3], "min_samples_leaf": [1, 5],
                      "class_weight": [None, "balanced_subsample"]}),
    "hgb": (build_hgb, {"learning_rate": [0.05, 0.1], "max_leaf_nodes": [15, 31],
                        "class_weight": [None, "balanced"]}),
}

In [ ]:
X = data[FEATURES]
y = data["label"]
fold = data["fold"]
is_test = fold.eq(TEST_FOLD).to_numpy()
is_val = fold.eq(VAL_FOLD).to_numpy()
is_train = ~(is_test | is_val)
print(f"{len(FEATURES)} features | train {is_train.sum():,} · validation {is_val.sum():,} · test {is_test.sum():,}")

### 4.1 Hyper-parameter tuning on the validation fold

In [ ]:
tuning = tune_on_validation(MODELS, X, y, is_train, is_val)
BEST = best_params(tuning)

view = tuning.assign(params=tuning["params"].astype(str))[["model", "params", "macro_f1", "f1_rainfed", "f1_irrigated", "balanced_accuracy"]]
display(view.groupby("model").head(3).round(3).set_index(["model", "params"]))
print("Selected:", BEST)

### 4.2 Final test (fold 0)

Each model is refitted with its selected parameters on folds 1–4 and scored once on the held-out test fold. Confidence intervals resample whole 20 km blocks, because parcels within a block are not independent.

In [ ]:
fitted, test_pred, test_ci = {}, {}, {}
for name, (build, _) in MODELS.items():
    model = build(**BEST[name]).fit(X[~is_test], y[~is_test])
    y_hat, p = predict(model, X[is_test])
    fitted[name] = model
    test_pred[name] = pd.DataFrame({"y_true": y[is_test], "y_pred": y_hat, "p_rainfed": p,
                                    "block_id": data.loc[is_test, "block_id"]})
    test_ci[name] = block_bootstrap(y[is_test], y_hat, p, data.loc[is_test, "block_id"], n_boot=N_BOOT, seed=SEED)

def fmt(ci):
    return ci.apply(lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}–{r.ci_high:.3f}]", axis=1)

test_table = pd.DataFrame({m: fmt(ci) for m, ci in test_ci.items()}).loc[HEADLINE]
test_table

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.6 * len(MODELS), 4.2), dpi=120)
for ax, (name, pred) in zip(np.atleast_1d(axes), test_pred.items()):
    plot_confusion(pred["y_true"], pred["y_pred"], ax, title=f"{name} — macro-F1 {test_ci[name].loc['macro_f1', 'estimate']:.3f}")
fig.suptitle(f"{RUN_NAME}: test fold (n = {is_test.sum():,})", fontweight="bold")
fig.tight_layout()
fig.savefig(MODEL_DIR / "test_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.3 Spatial vs random cross-validation

Every fold is held out once. The tuned parameters are reused unchanged; they were chosen on fold 1, so fold 1's spatial score is slightly optimistic.

In [ ]:
cv_rows = []
for name, (build, _) in MODELS.items():
    for scheme, col in [("spatial", "fold"), ("random", "random_fold")]:
        per_fold, _ = cross_validate(lambda: build(**BEST[name]), X, y, data[col])
        cv_rows.append(per_fold.reset_index().assign(model=name, cv=scheme))
cv = pd.concat(cv_rows, ignore_index=True)

cv_summary = (cv.groupby(["model", "cv"])[["macro_f1", "f1_rainfed", "f1_irrigated", "overall_accuracy"]]
                .agg(["mean", "std"]).round(3))
gap = (cv.groupby(["model", "cv"])["macro_f1"].mean().unstack()
         .assign(optimism=lambda t: t["random"] - t["spatial"]).round(3))
display(cv_summary)
gap.rename(columns={"optimism": "random − spatial (macro-F1)"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
plot_cv_comparison(cv, "macro_f1", axes[0])
plot_cv_comparison(cv, "f1_rainfed", axes[1])
axes[0].set_title("Macro-F1 per fold", fontweight="bold")
axes[1].set_title("Rainfed F1 per fold", fontweight="bold")
fig.suptitle(f"{RUN_NAME}: random vs spatial 5-fold CV", fontweight="bold")
fig.tight_layout()
fig.savefig(MODEL_DIR / "cv_random_vs_spatial.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Which variables and months matter

Monthly features are strongly correlated, so single-column permutation spreads importance thinly. Instead, *grouped* permutation importance shuffles all 12 months of one variable together (or all variables of one month) and measures the drop in macro-F1 of the random forest on the test fold.

In [ ]:
def grouped_permutation(model, X, y, groups: dict, n_repeats=10, seed=SEED) -> pd.DataFrame:
    """Drop in macro-F1 when the columns of each group are shuffled together (same row permutation)."""
    rng = np.random.default_rng(seed)
    base = f1_score(y, model.predict(X), average="macro")
    rows = []
    for name, cols in groups.items():
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[cols] = X[cols].to_numpy()[rng.permutation(len(X))]
            drops.append(base - f1_score(y, model.predict(Xp), average="macro"))
        rows.append({"group": name, "mean": np.mean(drops), "std": np.std(drops)})
    return pd.DataFrame(rows).set_index("group")

X_test, y_test = X[is_test], y[is_test]
by_variable = grouped_permutation(fitted["rf"], X_test, y_test,
                                  {v: [f"{v}_m{m:02d}" for m in range(1, 13)] for v in VARIABLES})
by_month = grouped_permutation(fitted["rf"], X_test, y_test,
                               {MONTHS[m - 1].strftime("%b"): [f"{v}_m{m:02d}" for v in VARIABLES] for m in range(1, 13)})

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=120)
bv = by_variable.sort_values("mean")
axes[0].barh(bv.index, bv["mean"], xerr=bv["std"], color="#5b8c5a")
axes[0].set_title("By variable (all months shuffled)", fontweight="bold")
axes[1].bar(by_month.index, by_month["mean"], yerr=by_month["std"], color="#4c78a8")
axes[1].set_title("By month (all variables shuffled)", fontweight="bold")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_xlabel("Drop in macro-F1")
axes[1].set_ylabel("Drop in macro-F1")
fig.tight_layout()
fig.savefig(MODEL_DIR / "grouped_permutation_importance.png", dpi=150, bbox_inches="tight")
plt.show()
pd.concat({"variable": by_variable, "month": by_month}).to_csv(MODEL_DIR / "grouped_permutation_importance.csv")

## 6. Save results

Everything needed to compare runs later is written to the run folder: the tuning table, test scores with CIs, per-fold CV scores, test predictions and the fitted models.

In [ ]:
tuning.assign(params=tuning["params"].astype(str)).to_csv(MODEL_DIR / "tuning_validation.csv", index=False)
pd.concat({m: ci for m, ci in test_ci.items()}, names=["model"]).to_csv(MODEL_DIR / "test_metrics.csv")
cv.to_csv(MODEL_DIR / "cv_folds.csv", index=False)
pd.concat(test_pred, names=["model"]).to_parquet(MODEL_DIR / "test_predictions.parquet")
for name, model in fitted.items():
    joblib.dump(model, MODEL_DIR / f"model_{name}.joblib")
(MODEL_DIR / "run_config.json").write_text(json.dumps({
    "run": RUN_NAME, "year": YEAR, "n_features": len(FEATURES), "n_parcels": int(len(data)),
    "test_fold": TEST_FOLD, "val_fold": VAL_FOLD, "seed": SEED, "best_params": BEST,
}, indent=2, default=str))
print("Saved to", MODEL_DIR.relative_to(ROOT))
sorted(p.name for p in MODEL_DIR.iterdir())